# Stream candidate with its fitted isochrone

The same steps as `candidate_investigation_example.ipynb`, with the isochrone and distance modulus fitted for the candidate (`data/candidates/candidate_fits.csv`: `mu`, `age` in Gyr, `z`, `N`, each with its lower and upper errors). The Z of the fits are on the Dotter2016 scale: the isochrone model is set to `"Dotter2016"` explicitly, so the match filter and the isochrone track use the fitted Z as it is.

No scan over distance moduli: one map, at the fitted distance modulus, to check the fit.

Steps:
1. inputs: the fitted values of the candidate;
2. catalog and stars of the window;
3. matched-filter map at the fitted distance modulus;
4. width;
5. on and off regions, depth and dust maps;
6. Hess diagram (on − off) with the fitted isochrone;
7. null test: Hess diagram of the regions moved off the stream;
8. summary;
9. GIF over distance moduli around the fitted one (`BUILD_GIF`).

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation
from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.stream_candidates import (
    CANDIDATE_FITS_PATH,
    write_candidate_fit_endpoints,
    write_candidate_width,
)
from streameucliddes.plotting.plot_candidate_investigation import (
    draw_stream_regions,
    plot_candidate_depth_dust_maps,
    plot_candidate_hess,
    plot_candidate_map,
    plot_phi1_profile,
    plot_phi2_profile,
    plot_width_fit_map,
)
from streameucliddes.plotting.plot_cmd import plot_hess, plot_isochrone, plot_match_filter_polygon
from streameucliddes.plotting.plot_style import escape_underscores
from streameucliddes.stream_frames import shift_endpoints_phi2

## 1. Inputs

The candidate by its id, and its row of the fits. The endpoints come from the list of candidates (`candidates_info.csv`), the distance modulus, age and Z from the fits. Figures go to the folder of the candidate with the suffix `_fitted_isochrone`, beside those of `candidate_investigation_example`. `from_candidate_id(source="fitted")`: the endpoints of the fitted track when the fits have them (`endpoint_1_ra`, ... written in section 4), else those of the list, picked by eye (a warning).

In [ ]:
CANDIDATE_ID = 17  # a candidate of candidate_fits.csv (2 or 6 for now)

FITS_PATH = CANDIDATE_FITS_PATH
ISOCHRONE_MODEL = "Dotter2016"  # the scale of the fitted Z
CATALOG = "euclid_dr1"
STAR_SELECTION = "morphology_cuts"
DUST_CORRECTION = "PLANCK"  # dust correction of the magnitudes, "SFD98" or "PLANCK"; also the dust map of the depth and dust maps
LOAD_MARGIN_DEG = 1.0  # stars read in the (ra, dec) box of the window plus this margin (the endpoints do not move here)
FIGURE_SUFFIX = "_fitted_isochrone"

fits = pd.read_csv(FITS_PATH).set_index("candidate")
fit_values = fits.loc[CANDIDATE_ID]
print(f"Candidate {CANDIDATE_ID}, fitted values:")
for name in ("mu", "age", "z", "N"):
    print(f"  {name:>3} = {fit_values[name]:.4g}  -{fit_values[name + '_err_lo']:.2g} +{fit_values[name + '_err_hi']:.2g}")

cand = CandidateInvestigation.from_candidate_id(
    CANDIDATE_ID,
    source="fitted",  # endpoints and distance modulus of the fits, else those of the list (by eye), with a warning
    fits_path=FITS_PATH,
    age=float(fit_values["age"]),
    metallicity=float(fit_values["z"]),
    isochrone_model=ISOCHRONE_MODEL,
    distance_moduli=[float(fit_values["mu"])],  # only the fitted one: no scan
    match_filter=get_match_filter(),
)
if CANDIDATE_ID in [3]:
    cand.padding_phi2_deg = 7
if CANDIDATE_ID == 11:
    cand.padding_phi2_deg = 9
print(f"{cand.name}: endpoints {cand.endpoint_1} and {cand.endpoint_2} deg, dm {cand.distance_modulus:.2f}")
cand.isochrone

## 2. Catalog and stars of the window

Euclid DR1 south in the (ra, dec) box of the window plus a small margin, then the stars of the window prepared with the fitted isochrone (`cand.prepare`).

In [ ]:
%%time
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
ra_range = ((ra_min - LOAD_MARGIN_DEG) % 360.0, (ra_max + LOAD_MARGIN_DEG) % 360.0)  # ra_min > ra_max: across ra = 0
dec_range = (max(dec_min - LOAD_MARGIN_DEG, -90.0), min(dec_max + LOAD_MARGIN_DEG, 90.0))
print(f"Loaded box: ra {ra_range[0]:.1f} to {ra_range[1]:.1f} deg, dec {dec_range[0]:.1f} to {dec_range[1]:.1f} deg")

catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range, dec_range=dec_range,
                           dust_correction=DUST_CORRECTION)
cand.prepare(catalog)

## 3. Matched-filter map at the fitted distance modulus

One map in the frame (`cand.compute_zoom_map`) with the fitted isochrone, at the fitted distance modulus.

In [ ]:
if CANDIDATE_ID == 11:
    phi1_range = (-4, 2.5)
elif CANDIDATE_ID == 1:
    phi1_range = (-4, 4.)
else:
    phi1_range = None
phi2_range = (-2, 2) if CANDIDATE_ID in [1, 7] else None

cand.compute_zoom_map(cand.distance_modulus)
fig, ax = plot_candidate_map(cand, phi1_range=phi1_range, phi2_range=phi2_range)
os.makedirs(cand.figure_folder, exist_ok=True)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}{FIGURE_SUFFIX}_map.png"), dpi=150)

## 4. Width

Stars in the match filter of the fitted isochrone at the fitted distance modulus → phi2 density profile → Gaussian plus a linear background (`fit_stream_width`), then the same stars in 2D against the model. The fitted sigma and its error are written in the row of the candidate in `candidate_fits.csv` (`sigma_deg`, `sigma_deg_err`), only if not there yet unless `force=True`. When the fits already have the fitted endpoints and the sigma (`cand.fitted_sigma_deg`, set by `from_candidate_id`), no fit: the regions (section 5) use phi2 = 0 and that sigma.

In [ ]:
if cand.fitted_sigma_deg is None:  # no fitted endpoints and sigma in the fits: fit the width
    cand.fit_width()  # background_degree=0 for a constant; phi1_range=..., sigma_guess=... to tune
    fit = cand.width_fit
    print(f"phi2_centre = {fit['phi2_centre']:.3f} ± {fit['errors']['phi2_centre']:.3f} deg, "
          f"sigma = {fit['sigma']:.3f} ± {fit['errors']['sigma']:.3f} deg, chi2 / n_dof = {fit['chi2']:.0f} / {fit['n_dof']}")
    # sigma_deg and sigma_deg_err in candidate_fits.csv; a width already there is kept unless force=True
    write_candidate_width(CANDIDATE_ID, cand.width_fit, FITS_PATH, force=False)
    fig, ax = plot_phi2_profile(cand.phi2_profile, cand.width_fit)
else:
    print(f"Fitted endpoints and sigma in the fits: no width fit, track at phi2 = 0, sigma = {cand.fitted_sigma_deg:.3f} deg")

In [ ]:
if cand.width_fit is not None:
    fig, axes = plot_width_fit_map(cand.width_map, cand.width_fit)

### Endpoints on the fitted track

The frame comes from the endpoints (those of the list, picked by eye, the first time): the fitted `phi2_centre` is the offset of the track from them. Moved by that offset in phi2 (`shift_endpoints_phi2`), the endpoints are on the track, and in their frame the track is at phi2 = 0: the centre of a later fit can be fixed at 0. `RECENTRE_ENDPOINTS = True` writes them in the row of the candidate in `candidate_fits.csv` (`write_candidate_fit_endpoints`: columns `endpoint_1_ra`, `endpoint_1_dec`, `endpoint_2_ra`, `endpoint_2_dec`), the endpoints of the list (`candidates_info.csv`) are kept as picked by eye. Then set it back to `False` and run the notebook again from the top: the frame of the fitted endpoints (section 1) and the sigma fitted above, with no new fit. Only after a width fit (not when the fits already have both).

In [ ]:
RECENTRE_ENDPOINTS = False  # True: the endpoints moved onto the fitted track written in candidate_fits.csv

if cand.width_fit is not None:
    phi2_centre, phi2_centre_error = fit["phi2_centre"], fit["errors"]["phi2_centre"]
    new_endpoint_1, new_endpoint_2 = shift_endpoints_phi2(cand.endpoint_1, cand.endpoint_2, phi2_centre)
    print(f"Track at phi2 = {phi2_centre:.3f} ± {phi2_centre_error:.3f} deg from the endpoints ({phi2_centre / fit['sigma']:.2f} sigma)")
    print(f"Endpoints on the track: ({new_endpoint_1[0]:.4f}, {new_endpoint_1[1]:.4f}) and "
          f"({new_endpoint_2[0]:.4f}, {new_endpoint_2[1]:.4f}) deg")
    if RECENTRE_ENDPOINTS:
        write_candidate_fit_endpoints(CANDIDATE_ID, new_endpoint_1, new_endpoint_2, FITS_PATH)
        print(f"Written in {FITS_PATH}: set RECENTRE_ENDPOINTS = False and run the notebook again from the top")

## 5. On and off regions

On region (`cand.define_regions()`: phi2_centre and sigma of the width fit, else phi2 = 0 and `cand.fitted_sigma_deg`): |phi2 − phi2_centre| ≤ `cand.on_n_sigma` σ; off regions: two strips below and above the track at `cand.off_n_sigma` σ; the off density is the mean of their two densities. Then the phi1 profile of on − off, to check where the stream starts and stops (`cand.define_regions(phi1_range=...)` to change it).

In [ ]:
cand.define_regions()
print(f"Areas outside the masks: on {cand.areas['on']:.2f} deg², off {cand.areas['off']:.2f} deg²")
fig, ax = plt.subplots(layout="constrained")
plot_candidate_map(cand, ax=ax, phi1_range=phi1_range, phi2_range=phi2_range)
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}{FIGURE_SUFFIX}_onoff.png"), dpi=150)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = plot_phi1_profile(cand.phi1_profile, cand.regions, ax=ax)

### Depth and dust maps

Does the candidate follow a fluctuation of the survey depth or of the dust? The Euclid depth maps of each band (limiting magnitude, `data/depth_maps/euclid`) and the E(B−V) map of the dust correction of the magnitudes (`DUST_CORRECTION`: SFD98 or Planck 2013), on the grid of the maps in the frame (`cand.compute_depth_dust_maps`), drawn beside the matched-filter map with the track and the regions (`plot_candidate_depth_dust_maps`), saved in the `cross_check` folder of the candidate (`cand.cross_check_folder`). `depth_quantities=("MAGLIM", "FRACDET")` adds the detection fraction of each band (holes in the coverage); `plot_candidate_depth_dust_map(cand, cand.depth_dust_maps["EBV"])` draws one map alone.

In [ ]:
cand.compute_depth_dust_maps(dust_correction=DUST_CORRECTION)  # depth_quantities=("MAGLIM", "FRACDET"): detection fraction too
fig, axes = plot_candidate_depth_dust_maps(cand, phi1_range=phi1_range, phi2_range=phi2_range)  # other_tracks=cand.other_tracks_phi12()
os.makedirs(cand.cross_check_folder, exist_ok=True)  # figures/outputs/candidates/<id>/cross_check
fig.savefig(os.path.join(cand.cross_check_folder, f"{cand.name}{FIGURE_SUFFIX}_depth_dust.png"), dpi=150)

## 6. Hess diagram with the fitted isochrone

On, off and on − off Hess diagrams in stars/deg²/bin (`compute_hess`), then the difference with the fitted isochrone (`compute_isochrone`: age, Z, model and distance modulus of the object) and the match-filter polygon.

In [ ]:
HESS_COLOR_BINS = np.arange(-1.0, 1.5 + 0.05, 0.1)  # VIS - H
HESS_MAG_BINS = np.arange(18.0, 26.0 + 0.125, 0.25)  # VIS
cand.compute_hess(color_bins=HESS_COLOR_BINS, mag_bins=HESS_MAG_BINS)
cand.compute_isochrone()

print(f"SNR {cand.summary()['snr']:.2f}")
fig, ax = plot_candidate_hess(cand, match_filter=True)
ax.text(0.97, 0.03, f"SNR = {cand.summary()['snr']:.1f}", transform=ax.transAxes, ha="right", va="bottom",
        bbox={"facecolor": "white", "edgecolor": "0.5", "alpha": 0.8})  # SNR in the lower right corner
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}{FIGURE_SUFFIX}_hess.png"), dpi=150)
candidate_clim = ax.images[0].get_clim()  # colour scale of the candidate, for the null test

## 7. Null test

The on region and the off region on its side moved off the stream (`compute_null_test`, see `candidate_investigation_example`): its on − off Hess diagram should show no isochrone and its SNR be about 0. Drawn on its own, on the colour scale of the candidate's diagram above, with the same isochrone and match filter.

In [ ]:
cand.compute_null_test(color_bins=HESS_COLOR_BINS, mag_bins=HESS_MAG_BINS)
null_test = cand.null_test
print(f"Null test: regions shifted {null_test['direction']} by {null_test['shift_deg']:.2f} deg; "
      f"SNR {null_test['snr']:.2f} ({null_test['n_excess']:.0f} ± {null_test['n_excess_error']:.0f} excess stars)")


In [ ]:
# The shifted regions of the null test on the map
fig, ax = plt.subplots(layout="constrained")
plot_candidate_map(cand, ax=ax, show_regions=False, phi1_range=phi1_range, phi2_range=phi2_range)
draw_stream_regions(ax, null_test["regions"])
ax.legend(loc="upper right", fontsize="small", framealpha=0.8)
ax.set_title(f"Null test: regions shifted {null_test['direction']} by {null_test['shift_deg']:.2f} deg")

In [ ]:

fig, ax = plot_hess(null_test["diff_hess"], difference=True, smoothing_bins=1.0, vlim=max(np.abs(candidate_clim)),
                   title=f"{escape_underscores(cand.name)}: null test, on $-$ off, SNR {null_test['snr']:.1f}")
plot_isochrone(ax, cand.isochrone_track)
plot_match_filter_polygon(ax, cand.distance_modulus, cand.isochrone, cand.search.match_filter)
ax.legend(loc="upper left", fontsize="small", framealpha=0.8)
ax.text(0.97, 0.03, f"SNR = {null_test['snr']:.1f}", transform=ax.transAxes, ha="right", va="bottom",
        bbox={"facecolor": "white", "edgecolor": "0.5", "alpha": 0.8})  # SNR in the lower right corner
fig.savefig(os.path.join(cand.figure_folder, f"{cand.name}{FIGURE_SUFFIX}_null_test_hess.png"), dpi=150)

## 8. Summary

In [ ]:
for key, value in cand.summary().items():
    print(f"{key:>15}: {value}")

## 9. GIF over distance moduli

`build_gif(cand)`: the map in the frame at each distance modulus around the fitted one (±1 every 0.1 by default: `dm_half_range`, `dm_step`), with the fitted isochrone, the track of the candidate and, in another colour, the known streams and the other listed candidates in the window (`cand.other_tracks_phi12()`). The frames go to `cand.figure_folder / "frames"`, the GIF to `cand.figure_folder`, their names with `FIGURE_SUFFIX` (`file_prefix`): those of `candidate_investigation_example` are kept. `BUILD_GIF = False` skips it.

In [ ]:
BUILD_GIF = False  # False: no GIF (it computes one map per frame)

In [ ]:
if BUILD_GIF:
    from IPython.display import Image, display

    from streameucliddes.plotting.plot_candidate_investigation import build_gif

    # The same view as the maps of section 3 (phi1_range, phi2_range); vmin and vmax for one colour scale over the GIF
    gif_path, frame_paths = build_gif(cand, file_prefix=f"{cand.name}{FIGURE_SUFFIX}",
                                      phi1_range=phi1_range, phi2_range=phi2_range)
    print(f"{len(frame_paths)} frames in {frame_paths[0].parent}")
    display(Image(filename=str(gif_path)))